In [2]:
!pip install pandas

In [3]:
!pip install numpy
!pip install matplotlib
!pip install seaborn
!pip install scikit-learn

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("🎸 Som testado e aprovado! O ambiente tcc_energia está 100% online!")
print(f"Versão do Pandas: {pd.__version__}")
print(f"Versão do Numpy: {np.__version__}")
print("Aguardando coordenadas para iniciar o download dos dados da CCEE e ONS...")

🎸 Som testado e aprovado! O ambiente tcc_energia está 100% online!
Versão do Pandas: 2.3.3
Versão do Numpy: 2.2.6
Aguardando coordenadas para iniciar o download dos dados da CCEE e ONS...


In [5]:
import pandas as pd
import os

# Caminho para o nosso arquivo de reconhecimento (ajuste o nome exato do CSV de 2021)
caminho_arquivo = r'C:\Users\felip\OneDrive\Desktop\DSP - Data Science Project\Data Science & Analytics\0. TCC\Codes_Project\Data\Raw\pld_horario_2021.csv'

print(f"Iniciando varredura no arquivo: {caminho_arquivo}...")

try:
    # Arquivos da CCEE e ONS geralmente usam separador ';' e decimais com vírgula.
    # O encoding 'latin-1' é o feitiço contra caracteres bizarros em palavras acentuadas.
    df_reconhecimento = pd.read_csv(caminho_arquivo, sep=';', decimal=',', encoding='latin-1')
    
    print("✅ Leitura bem-sucedida! Matriz decodificada.")
    print("-" * 60)
    print("🛠️ Colunas identificadas pelo Pandas:")
    print(df_reconhecimento.columns.tolist())
    print("-" * 60)
    
    # Exibe as 5 primeiras linhas com uma formatação bonita do Jupyter
    display(df_reconhecimento.head())

except FileNotFoundError:
    print("🚨 Arquivo não encontrado! Verifique se o nome e o caminho estão idênticos ao da pasta.")
except Exception as e:
    print(f"🚨 Erro inesperado na Matrix: {e}")

Iniciando varredura no arquivo: C:\Users\felip\OneDrive\Desktop\DSP - Data Science Project\Data Science & Analytics\0. TCC\Codes_Project\Data\Raw\pld_horario_2021.csv...
✅ Leitura bem-sucedida! Matriz decodificada.
------------------------------------------------------------
🛠️ Colunas identificadas pelo Pandas:
['MES_REFERENCIA', 'SUBMERCADO', 'PERIODO_COMERCIALIZACAO', 'DIA', 'HORA', 'PLD_HORA']
------------------------------------------------------------


,MES_REFERENCIA,SUBMERCADO,PERIODO_COMERCIALIZACAO,DIA,HORA,PLD_HORA
0,202101,NORDESTE,1,1,0,204.36
1,202101,NORTE,1,1,0,204.37
2,202101,SUDESTE,1,1,0,204.37
3,202101,SUL,1,1,0,204.37
4,202101,NORDESTE,2,1,1,206.82


In [6]:
import pandas as pd
import glob
import os

print("🏗️ Iniciando a construção da torre de dados do PLD...")

# 1. Mapeando todos os arquivos de PLD na pasta Raw (ajuste o caminho se necessário)
caminho_arquivos = r"C:\Users\felip\OneDrive\Desktop\DSP - Data Science Project\Data Science & Analytics\0. TCC\Codes_Project\Data\Raw\pld*.csv" 
arquivos = glob.glob(caminho_arquivos)

if not arquivos:
    print("🚨 Nenhum arquivo encontrado! Verifique se os arquivos começam com 'pld' e estão na pasta Data/Raw.")
else:
    lista_dfs = []

    # 2. Ler e empilhar todos os anos
    for arquivo in arquivos:
        print(f"📖 Lendo e processando: {os.path.basename(arquivo)}")
        df_temp = pd.read_csv(arquivo, sep=';', decimal=',', encoding='latin-1')
        lista_dfs.append(df_temp)

    # Junta todos os anos num super DataFrame
    df_pld = pd.concat(lista_dfs, ignore_index=True)

    # 3. O Filtro de Ouro: Focar apenas no motor industrial do Brasil
    df_pld = df_pld[df_pld['SUBMERCADO'] == 'SUDESTE'].copy()

    # 4. Magia do Tempo (Engenharia de Features Temporais)
    # Quebrando a coluna 202101 em Ano (2021) e Mês (01)
    df_pld['ANO'] = df_pld['MES_REFERENCIA'].astype(str).str[:4].astype(int)
    df_pld['MES'] = df_pld['MES_REFERENCIA'].astype(str).str[4:6].astype(int)
    df_pld['DIA'] = df_pld['DIA'].astype(int)

    # A CCEE já entrega a hora no padrão 0-23, então apenas convertemos para número inteiro
    df_pld['HORA_PYTHON'] = df_pld['HORA'].astype(int)

    # Fundindo tudo na linha do tempo oficial
    df_pld['DATA_HORA'] = pd.to_datetime(
        dict(year=df_pld['ANO'], month=df_pld['MES'], day=df_pld['DIA'], hour=df_pld['HORA_PYTHON'])
    )

    # 5. Lapidação: Transformar a Data no Índice e manter só o Preço
    df_pld = df_pld.set_index('DATA_HORA')
    df_pld = df_pld[['PLD_HORA']].rename(columns={'PLD_HORA': 'PLD'})

    # Ordenar cronologicamente (do dia 01/01/2021 até o último dia baixado)
    df_pld = df_pld.sort_index()

    print("-" * 60)
    print("✅ Consolidação Concluída com Sucesso!")
    print(f"Total de horas registradas no modelo: {len(df_pld)} linhas (variável Y garantida).")
    display(df_pld.head(10))

🏗️ Iniciando a construção da torre de dados do PLD...
📖 Lendo e processando: pld_horario_2021.csv
📖 Lendo e processando: pld_horario_2022.csv
📖 Lendo e processando: pld_horario_2023.csv
📖 Lendo e processando: pld_horario_2024.csv
📖 Lendo e processando: pld_horario_2025.csv
------------------------------------------------------------
✅ Consolidação Concluída com Sucesso!
Total de horas registradas no modelo: 43824 linhas (variável Y garantida).


,PLD
DATA_HORA,
2021-01-01 00:00:00,204.37
2021-01-01 01:00:00,206.82
2021-01-01 02:00:00,204.61
2021-01-01 03:00:00,209.2
2021-01-01 04:00:00,207.19
2021-01-01 05:00:00,201.99
2021-01-01 06:00:00,195.53
2021-01-01 07:00:00,195.4
2021-01-01 08:00:00,195.5


In [8]:
import pandas as pd
import os

# Caminho para o arquivo de carga do ONS (ajuste o nome exato)
caminho_carga = r"C:\Users\felip\OneDrive\Desktop\DSP - Data Science Project\Data Science & Analytics\0. TCC\Codes_Project\Data\Raw\CURVA_CARGA_2021.csv" 

print(f"🔎 Iniciando varredura no arquivo do ONS: {caminho_carga}...")

try:
    # ONS costuma usar separador ';'
    # Tentamos utf-8 primeiro, que é o padrão de dados abertos mais moderno
    df_carga_recon = pd.read_csv(caminho_carga, sep=';', encoding='utf-8')
    
    print("✅ Leitura bem-sucedida! Matriz decodificada.")
    print("-" * 60)
    print("🛠️ Colunas identificadas pelo Pandas:")
    print(df_carga_recon.columns.tolist())
    print("-" * 60)
    
    # Vamos descobrir como o ONS chama o submercado Sudeste
    # A coluna costuma se chamar 'nom_subsistema' ou 'id_subsistema'
    coluna_sub = [col for col in df_carga_recon.columns if 'sub' in col.lower()]
    if coluna_sub:
        nome_col = coluna_sub[0]
        print(f"🌍 Regiões encontradas na coluna '{nome_col}':")
        print(df_carga_recon[nome_col].unique())
    
    display(df_carga_recon.head())

except FileNotFoundError:
    print("🚨 Arquivo não encontrado! Verifique o nome e a pasta.")
except UnicodeDecodeError:
    print("🚨 Erro de encoding! Tente trocar 'utf-8' por 'latin-1' no pd.read_csv")
except Exception as e:
    print(f"🚨 Erro inesperado: {e}")

🔎 Iniciando varredura no arquivo do ONS: C:\Users\felip\OneDrive\Desktop\DSP - Data Science Project\Data Science & Analytics\0. TCC\Codes_Project\Data\Raw\CURVA_CARGA_2021.csv...
✅ Leitura bem-sucedida! Matriz decodificada.
------------------------------------------------------------
🛠️ Colunas identificadas pelo Pandas:
['id_subsistema', 'nom_subsistema', 'din_instante', 'val_cargaenergiahomwmed']
------------------------------------------------------------
🌍 Regiões encontradas na coluna 'id_subsistema':
['N' 'NE' 'S' 'SE']


,id_subsistema,nom_subsistema,din_instante,val_cargaenergiahomwmed
0,N,NORTE,2021-01-01 00:00:00,5511.801000
1,NE,NORDESTE,2021-01-01 00:00:00,10343.540000
2,S,SUL,2021-01-01 00:00:00,9060.566986
3,SE,SUDESTE,2021-01-01 00:00:00,33498.352000
4,N,NORTE,2021-01-01 01:00:00,5470.133000


In [9]:
import pandas as pd
import glob
import os

print("🏗️ Iniciando a fusão do motor de Carga com o PLD...")

# 1. Mapear todos os arquivos da Curva de Carga
caminho_arquivos_carga = r"Data\Raw\CURVA_CARGA_*.csv"
arquivos_carga = glob.glob(caminho_arquivos_carga)

lista_dfs_carga = []

# 2. Ler e empilhar todos os anos
for arquivo in arquivos_carga:
    print(f"📖 Lendo e processando: {os.path.basename(arquivo)}")
    df_temp = pd.read_csv(arquivo, sep=';', encoding='utf-8')
    lista_dfs_carga.append(df_temp)

# Junta todos os anos num super DataFrame da Carga
df_carga = pd.concat(lista_dfs_carga, ignore_index=True)

# 3. O Filtro de Ouro: Focar apenas no Sudeste
df_carga = df_carga[df_carga['nom_subsistema'] == 'SUDESTE'].copy()

# 4. Magia do Tempo (O ONS já entregou mastigado)
df_carga['DATA_HORA'] = pd.to_datetime(df_carga['din_instante'])

# 5. Lapidação: Transformar a Data no Índice e manter só a Carga
df_carga = df_carga.set_index('DATA_HORA')
df_carga = df_carga[['val_cargaenergiahomwmed']].rename(columns={'val_cargaenergiahomwmed': 'CARGA_MWMED'})

# Ordenar cronologicamente por precaução
df_carga = df_carga.sort_index()

# 6. O GRANDE ENCONTRO: Unindo PLD (Y) e Carga (X)
# Usamos o 'join' do Pandas, que alinha perfeitamente pelo índice (DATA_HORA)
df_mestre = df_pld.join(df_carga, how='left')

print("-" * 60)
print("✅ Fusão Concluída! O (Y) e o primeiro (X) estão sincronizados.")
print(f"Total de horas registradas: {len(df_mestre)}")
print(f"Valores de Carga faltando (Buracos do ONS): {df_mestre['CARGA_MWMED'].isna().sum()}")
print("-" * 60)
display(df_mestre.head(10))

🏗️ Iniciando a fusão do motor de Carga com o PLD...
📖 Lendo e processando: CURVA_CARGA_2021.csv
📖 Lendo e processando: CURVA_CARGA_2022.csv
📖 Lendo e processando: CURVA_CARGA_2023.csv
📖 Lendo e processando: CURVA_CARGA_2024.csv
📖 Lendo e processando: CURVA_CARGA_2025.csv
------------------------------------------------------------
✅ Fusão Concluída! O (Y) e o primeiro (X) estão sincronizados.
Total de horas registradas: 43824
Valores de Carga faltando (Buracos do ONS): 0
------------------------------------------------------------


,PLD,CARGA_MWMED
DATA_HORA,,
2021-01-01 00:00:00,204.37,33498.352
2021-01-01 01:00:00,206.82,33183.006
2021-01-01 02:00:00,204.61,32630.701
2021-01-01 03:00:00,209.2,31738.470
2021-01-01 04:00:00,207.19,31233.243
2021-01-01 05:00:00,201.99,29944.329
2021-01-01 06:00:00,195.53,28583.587
2021-01-01 07:00:00,195.4,28589.812
2021-01-01 08:00:00,195.5,29052.180


In [10]:
import pandas as pd

# Ajuste o nome exato do seu arquivo de ENA de 2021
caminho_ena = r"C:\Users\felip\OneDrive\Desktop\DSP - Data Science Project\Data Science & Analytics\0. TCC\Codes_Project\Data\Raw\ENA_DIARIO_REE_2021.csv" 

print(f"🔎 Analisando a 'chuva' no arquivo: {caminho_ena}...")

try:
    # Testando latin-1 pois arquivos de ENA costumam ter acentos nos nomes das bacias
    df_ena_recon = pd.read_csv(caminho_ena, sep=';', encoding='latin-1')
    
    print("✅ Leitura bem-sucedida!")
    print("-" * 60)
    print("🛠️ Colunas identificadas:")
    print(df_ena_recon.columns.tolist())
    print("-" * 60)
    
    # Verificando as regiões (Subsistemas)
    col_sub = [col for col in df_ena_recon.columns if 'sub' in col.lower()]
    if col_sub:
        print(f"🌍 Regiões na coluna '{col_sub[0]}':")
        print(df_ena_recon[col_sub[0]].unique())
        
    display(df_ena_recon.head())

except Exception as e:
    print(f"🚨 Erro na Matrix: {e}")

🔎 Analisando a 'chuva' no arquivo: C:\Users\felip\OneDrive\Desktop\DSP - Data Science Project\Data Science & Analytics\0. TCC\Codes_Project\Data\Raw\ENA_DIARIO_REE_2021.csv...
✅ Leitura bem-sucedida!
------------------------------------------------------------
🛠️ Colunas identificadas:
['nom_reservatorioee', 'ena_data', 'ena_bruta_ree_mwmed', 'ena_bruta_ree_percentualmlt', 'ena_armazenavel_ree_mwmed', 'ena_armazenavel_ree_percentualmlt']
------------------------------------------------------------


,nom_reservatorioee,ena_data,ena_bruta_ree_mwmed,ena_bruta_ree_percentualmlt,ena_armazenavel_ree_mwmed,ena_armazenavel_ree_percentualmlt
0,BELO MONTE,2021-01-01,1405.686000,24.9711,1405.686000,24.9711
1,IGUACU,2021-01-01,3882.294000,94.3457,3794.588000,92.2143
2,ITAIPU,2021-01-01,3047.047068,90.6298,3047.047068,90.6298
3,MADEIRA,2021-01-01,5484.850000,66.9702,5457.045000,66.6307
4,MANAUS-AMAPA,2021-01-01,537.693000,109.2704,537.693000,109.2704


In [11]:
import pandas as pd

# Ajuste o nome exato do seu arquivo de ENA de 2021
caminho_ena = r"C:\Users\felip\OneDrive\Desktop\DSP - Data Science Project\Data Science & Analytics\0. TCC\Codes_Project\Data\Raw\ENA_DIARIO_SUBSISTEMA_2021.csv"

print(f"🔎 Analisando a 'chuva' no arquivo: {caminho_ena}...")

try:
    # Testando latin-1 pois arquivos de ENA costumam ter acentos nos nomes das bacias
    df_ena_recon = pd.read_csv(caminho_ena, sep=';', encoding='latin-1')
    
    print("✅ Leitura bem-sucedida!")
    print("-" * 60)
    print("🛠️ Colunas identificadas:")
    print(df_ena_recon.columns.tolist())
    print("-" * 60)
    
    # Verificando as regiões (Subsistemas)
    col_sub = [col for col in df_ena_recon.columns if 'sub' in col.lower()]
    if col_sub:
        print(f"🌍 Regiões na coluna '{col_sub[0]}':")
        print(df_ena_recon[col_sub[0]].unique())
        
    display(df_ena_recon.head())

except Exception as e:
    print(f"🚨 Erro na Matrix: {e}")

🔎 Analisando a 'chuva' no arquivo: C:\Users\felip\OneDrive\Desktop\DSP - Data Science Project\Data Science & Analytics\0. TCC\Codes_Project\Data\Raw\ENA_DIARIO_SUBSISTEMA_2021.csv...
✅ Leitura bem-sucedida!
------------------------------------------------------------
🛠️ Colunas identificadas:
['id_subsistema', 'nom_subsistema', 'ena_data', 'ena_bruta_regiao_mwmed', 'ena_bruta_regiao_percentualmlt', 'ena_armazenavel_regiao_mwmed', 'ena_armazenavel_regiao_percentualmlt']
------------------------------------------------------------
🌍 Regiões na coluna 'id_subsistema':
['N' 'NE' 'S' 'SE']


,id_subsistema,nom_subsistema,ena_data,ena_bruta_regiao_mwmed,ena_bruta_regiao_percentualmlt,ena_armazenavel_regiao_mwmed,ena_armazenavel_regiao_percentualmlt
0,N,NORTE,2021-01-01,6104.027832,39.141800,6104.027832,39.141800
1,NE,NORDESTE,2021-01-01,5371.078125,39.751499,5356.062988,39.640400
2,S,SUL,2021-01-01,5893.043945,78.827202,5490.962891,73.448898
3,SE,SUDESTE,2021-01-01,41167.160156,62.561100,40896.367188,62.149601
4,N,NORTE,2021-01-02,6100.042969,39.116299,6100.042969,39.116299


In [12]:
import pandas as pd
import glob
import os

print("🌧️ Iniciando a fusão da ENA (Chuvas) com a Tabela Mestre...")

# 1. Mapear os arquivos de ENA (Ajuste o caminho se necessário)
caminho_arquivos_ena = r"Data\Raw\ENA_DIARIO_SUBSISTEMA_*.csv"
arquivos_ena = glob.glob(caminho_arquivos_ena)

lista_dfs_ena = []

# 2. Ler e empilhar
for arquivo in arquivos_ena:
    print(f"📖 Lendo e processando: {os.path.basename(arquivo)}")
    # Como você testou e deu certo, o utf-8 (padrão) deve funcionar sem o latin-1 aqui
    df_temp = pd.read_csv(arquivo, sep=';', encoding='utf-8') 
    lista_dfs_ena.append(df_temp)

df_ena = pd.concat(lista_dfs_ena, ignore_index=True)

# 3. Filtro de Ouro (SUDESTE)
df_ena = df_ena[df_ena['nom_subsistema'] == 'SUDESTE'].copy()

# 4. Preparar o Índice Diário
df_ena['DATA'] = pd.to_datetime(df_ena['ena_data'])
df_ena = df_ena.set_index('DATA')

# Isolando apenas a coluna que importa
df_ena = df_ena[['ena_bruta_regiao_mwmed']].rename(columns={'ena_bruta_regiao_mwmed': 'ENA_MWMED'})

# 5. A MÁGICA DA DOBRA DO TEMPO
# Criamos uma coluna temporária no df_mestre apenas com o "Dia" (ignorando a hora)
df_mestre['DATA_DIA'] = df_mestre.index.normalize()

# Fazemos o merge! Ele vai repetir o valor diário da ENA para todas as horas com a mesma data
df_mestre = df_mestre.reset_index()
df_mestre = df_mestre.merge(df_ena, left_on='DATA_DIA', right_index=True, how='left')

# Limpamos a bagunça temporária e voltamos a 'DATA_HORA' como a espinha dorsal
df_mestre = df_mestre.set_index('DATA_HORA')
df_mestre = df_mestre.drop(columns=['DATA_DIA'])

print("-" * 60)
print("✅ Fusão Concluída! Chuvas (ENA) alinhadas hora a hora.")
print(f"Valores de ENA faltando: {df_mestre['ENA_MWMED'].isna().sum()}")
print("-" * 60)

# Imprimindo 25 linhas para você ver a exata virada do dia 01 para o dia 02!
display(df_mestre.head(25))

🌧️ Iniciando a fusão da ENA (Chuvas) com a Tabela Mestre...
📖 Lendo e processando: ENA_DIARIO_SUBSISTEMA_2021.csv
📖 Lendo e processando: ENA_DIARIO_SUBSISTEMA_2022.csv
📖 Lendo e processando: ENA_DIARIO_SUBSISTEMA_2023.csv
📖 Lendo e processando: ENA_DIARIO_SUBSISTEMA_2024.csv
📖 Lendo e processando: ENA_DIARIO_SUBSISTEMA_2025.csv
------------------------------------------------------------
✅ Fusão Concluída! Chuvas (ENA) alinhadas hora a hora.
Valores de ENA faltando: 0
------------------------------------------------------------


,PLD,CARGA_MWMED,ENA_MWMED
DATA_HORA,,,
2021-01-01 00:00:00,204.37,33498.352,41167.160156
2021-01-01 01:00:00,206.82,33183.006,41167.160156
2021-01-01 02:00:00,204.61,32630.701,41167.160156
2021-01-01 03:00:00,209.2,31738.470,41167.160156
2021-01-01 04:00:00,207.19,31233.243,41167.160156
2021-01-01 05:00:00,201.99,29944.329,41167.160156
2021-01-01 06:00:00,195.53,28583.587,41167.160156
2021-01-01 07:00:00,195.4,28589.812,41167.160156
2021-01-01 08:00:00,195.5,29052.180,41167.160156


In [13]:
import pandas as pd

# Ajuste para o nome exato do seu arquivo EAR de 2021
caminho_ear = r"C:\Users\felip\OneDrive\Desktop\DSP - Data Science Project\Data Science & Analytics\0. TCC\Codes_Project\Data\Raw\EAR_DIARIO_SUBSISTEMA_2021.csv"

print(f"🔎 Inspecionando as baterias (EAR) em: {caminho_ear}...")

df_ear_recon = pd.read_csv(caminho_ear, sep=';', encoding='utf-8')
print("🛠️ Colunas identificadas:")
print(df_ear_recon.columns.tolist())
display(df_ear_recon.head(3))

🔎 Inspecionando as baterias (EAR) em: C:\Users\felip\OneDrive\Desktop\DSP - Data Science Project\Data Science & Analytics\0. TCC\Codes_Project\Data\Raw\EAR_DIARIO_SUBSISTEMA_2021.csv...
🛠️ Colunas identificadas:
['id_subsistema', 'nom_subsistema', 'ear_data', 'ear_max_subsistema', 'ear_verif_subsistema_mwmes', 'ear_verif_subsistema_percentual']


,id_subsistema,nom_subsistema,ear_data,ear_max_subsistema,ear_verif_subsistema_mwmes,ear_verif_subsistema_percentual
0,NE,NORDESTE,2021-01-01,52624.082031,24099.347656,45.795300
1,N,NORTE,2021-01-01,15302.396484,4376.652832,28.601101
2,SE,SUDESTE,2021-01-01,204560.843750,38768.015625,18.951799


In [14]:
import pandas as pd
import glob
import os

print("🔋 Iniciando a fusão do EAR (Nível dos Reservatórios) com a Tabela Mestre...")

# 1. Mapear os arquivos de EAR (Ajuste o caminho se necessário)
caminho_arquivos_ear = r"Data\Raw\EAR_DIARIO_SUBSISTEMA_*.csv"
arquivos_ear = glob.glob(caminho_arquivos_ear)

lista_dfs_ear = []

# 2. Ler e empilhar
for arquivo in arquivos_ear:
    print(f"📖 Lendo e processando: {os.path.basename(arquivo)}")
    df_temp = pd.read_csv(arquivo, sep=';', encoding='utf-8') 
    lista_dfs_ear.append(df_temp)

df_ear = pd.concat(lista_dfs_ear, ignore_index=True)

# 3. Filtro de Ouro (SUDESTE)
df_ear = df_ear[df_ear['nom_subsistema'] == 'SUDESTE'].copy()

# 4. Preparar o Índice Diário
df_ear['DATA'] = pd.to_datetime(df_ear['ear_data'])
df_ear = df_ear.set_index('DATA')

# Isolando a "Bateria" (Percentual)
df_ear = df_ear[['ear_verif_subsistema_percentual']].rename(columns={'ear_verif_subsistema_percentual': 'EAR_PERCENTUAL'})

# 5. A MÁGICA DA DOBRA DO TEMPO
df_mestre['DATA_DIA'] = df_mestre.index.normalize()

df_mestre = df_mestre.reset_index()
df_mestre = df_mestre.merge(df_ear, left_on='DATA_DIA', right_index=True, how='left')

df_mestre = df_mestre.set_index('DATA_HORA')
df_mestre = df_mestre.drop(columns=['DATA_DIA'])

print("-" * 60)
print("🏆 O TRITURADOR DE DADOS CONCLUIU SEU TRABALHO!")
print(f"Total de horas registradas: {len(df_mestre)}")
print(f"Radar de Falhas (Valores Faltantes):")
print(df_mestre.isna().sum())
print("-" * 60)

# 6. SALVANDO O OURO
# Criando a pasta Processed caso ela não exista
pasta_destino = r"Data\Processed"
os.makedirs(pasta_destino, exist_ok=True)

caminho_salvamento = os.path.join(pasta_destino, "dataset_mestre_pld.csv")
df_mestre.to_csv(caminho_salvamento)

print(f"💾 Base de dados perfeitamente limpa e salva em: {caminho_salvamento}")
display(df_mestre.head(10))

🔋 Iniciando a fusão do EAR (Nível dos Reservatórios) com a Tabela Mestre...
📖 Lendo e processando: EAR_DIARIO_SUBSISTEMA_2021.csv
📖 Lendo e processando: EAR_DIARIO_SUBSISTEMA_2022.csv
📖 Lendo e processando: EAR_DIARIO_SUBSISTEMA_2023.csv
📖 Lendo e processando: EAR_DIARIO_SUBSISTEMA_2024.csv
📖 Lendo e processando: EAR_DIARIO_SUBSISTEMA_2025.csv
------------------------------------------------------------
🏆 O TRITURADOR DE DADOS CONCLUIU SEU TRABALHO!
Total de horas registradas: 43824
Radar de Falhas (Valores Faltantes):
PLD               0
CARGA_MWMED       0
ENA_MWMED         0
EAR_PERCENTUAL    0
dtype: int64
------------------------------------------------------------
💾 Base de dados perfeitamente limpa e salva em: Data\Processed\dataset_mestre_pld.csv


,PLD,CARGA_MWMED,ENA_MWMED,EAR_PERCENTUAL
DATA_HORA,,,,
2021-01-01 00:00:00,204.37,33498.352,41167.160156,18.951799
2021-01-01 01:00:00,206.82,33183.006,41167.160156,18.951799
2021-01-01 02:00:00,204.61,32630.701,41167.160156,18.951799
2021-01-01 03:00:00,209.2,31738.470,41167.160156,18.951799
2021-01-01 04:00:00,207.19,31233.243,41167.160156,18.951799
2021-01-01 05:00:00,201.99,29944.329,41167.160156,18.951799
2021-01-01 06:00:00,195.53,28583.587,41167.160156,18.951799
2021-01-01 07:00:00,195.4,28589.812,41167.160156,18.951799
2021-01-01 08:00:00,195.5,29052.180,41167.160156,18.951799


In [1]:
import pandas as pd

print("⏳ Carregando o ouro bruto e iniciando a criação de Memória (Lags)...")

# 1. Carregar a base mestre salva na Fase 3
caminho_base = r"Data\Processed\dataset_mestre_pld.csv"
# parse_dates e index_col já transformam a coluna de data no índice automaticamente
df_modelo = pd.read_csv(caminho_base, index_col='DATA_HORA', parse_dates=True)

# 2. Definindo quantas horas no passado queremos olhar
# O ciclo diário é muito forte na energia. Vamos olhar as últimas 1, 2, 3 e 24 horas do PLD
lags_pld = [1, 2, 3, 24] 
# Para a carga, saber a de 1h atrás e a do mesmo horário ontem é essencial
lags_carga = [1, 24]

# Gerando as novas colunas
for lag in lags_pld:
    df_modelo[f'PLD_lag_{lag}'] = df_modelo['PLD'].shift(lag)

for lag in lags_carga:
    df_modelo[f'CARGA_lag_{lag}'] = df_modelo['CARGA_MWMED'].shift(lag)

# Para ENA e EAR (que eram constantes no dia), o lag de 24h representa o valor do "dia anterior"
df_modelo['ENA_lag_24'] = df_modelo['ENA_MWMED'].shift(24)
df_modelo['EAR_lag_24'] = df_modelo['EAR_PERCENTUAL'].shift(24)

# 3. O Preço da Viagem no Tempo
# Como não existe "dia anterior" para o dia 01/01/2021 (o início da base), 
# as primeiras 24 horas ficarão com valores vazios (NaN). Precisamos sacrificar essas linhas.
tamanho_original = len(df_modelo)
df_modelo = df_modelo.dropna()
tamanho_novo = len(df_modelo)

print("-" * 60)
print("✅ Memória temporal injetada com sucesso!")
print(f"Linhas sacrificadas para criar o passado: {tamanho_original - tamanho_novo}")
print(f"Novo total de linhas aptas para treino: {tamanho_novo}")
print("-" * 60)

# Mostrando as colunas a partir do 2º dia para ver os lags preenchidos
display(df_modelo.head())

⏳ Carregando o ouro bruto e iniciando a criação de Memória (Lags)...
------------------------------------------------------------
✅ Memória temporal injetada com sucesso!
Linhas sacrificadas para criar o passado: 24
Novo total de linhas aptas para treino: 43800
------------------------------------------------------------


,PLD,CARGA_MWMED,ENA_MWMED,EAR_PERCENTUAL,PLD_lag_1,PLD_lag_2,PLD_lag_3,PLD_lag_24,CARGA_lag_1,CARGA_lag_24,ENA_lag_24,EAR_lag_24
DATA_HORA,,,,,,,,,,,,
2021-01-02 00:00:00,276.93,33749.104,41696.214844,19.163,218.72,221.00,223.06,204.37,35584.782,33498.352,41167.160156,18.951799
2021-01-02 01:00:00,272.23,32224.655,41696.214844,19.163,276.93,218.72,221.00,206.82,33749.104,33183.006,41167.160156,18.951799
2021-01-02 02:00:00,270.09,31234.208,41696.214844,19.163,272.23,276.93,218.72,204.61,32224.655,32630.701,41167.160156,18.951799
2021-01-02 03:00:00,271.55,30674.752,41696.214844,19.163,270.09,272.23,276.93,209.20,31234.208,31738.470,41167.160156,18.951799
2021-01-02 04:00:00,271.47,30382.045,41696.214844,19.163,271.55,270.09,272.23,207.19,30674.752,31233.243,41167.160156,18.951799


In [2]:
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

print("✂️ Iniciando o Corte Cronológico e o Nivelamento (Sem Spoilers do Futuro)...")

# 1. A Linha de Corte (Train, Validation, Test)
# Treino: 2021 a 2023 (onde a IA estuda)
df_treino = df_modelo[df_modelo.index.year <= 2023].copy()

# Validação: 2024 (onde a IA faz simulados para calibrar)
df_validacao = df_modelo[df_modelo.index.year == 2024].copy()

# Teste: 2025 (a prova final, dados nunca vistos)
df_teste = df_modelo[df_modelo.index.year >= 2025].copy()

print(f"Dimensões do Corte -> Treino: {len(df_treino)} | Validação: {len(df_validacao)} | Teste: {len(df_teste)}")

# 2. O Nivelamento de Poder (MinMaxScaler)
# ATENÇÃO: Vamos criar DOIS scalers. Um para o X (variáveis exógenas e lags) e um para o Y (PLD atual)
# Precisamos do scaler do Y separado para conseguirmos "desnormalizar" a previsão lá no final do projeto para ver o valor em R$ reais.

alvo = ['PLD']
# Seleciona todas as colunas que NÃO são o PLD atual
features = [col for col in df_modelo.columns if col not in alvo]

scaler_X = MinMaxScaler()
scaler_Y = MinMaxScaler()

# 3. O Treinamento da Régua (FIT apenas no Treino!)
df_treino[features] = scaler_X.fit_transform(df_treino[features])
df_treino[alvo] = scaler_Y.fit_transform(df_treino[alvo])

# 4. Aplicando a régua no resto (Apenas TRANSFORM, sem dar fit de novo)
df_validacao[features] = scaler_X.transform(df_validacao[features])
df_validacao[alvo] = scaler_Y.transform(df_validacao[alvo])

df_teste[features] = scaler_X.transform(df_teste[features])
df_teste[alvo] = scaler_Y.transform(df_teste[alvo])

print("-" * 60)
print("✅ Divisão e Nivelamento concluídos com perfeição!")
print(f"O PLD agora varia matematicamente entre: {df_treino['PLD'].min()} e {df_treino['PLD'].max()}")
print("-" * 60)

# Mostrando o Treino normalizado
display(df_treino.head())

✂️ Iniciando o Corte Cronológico e o Nivelamento (Sem Spoilers do Futuro)...
Dimensões do Corte -> Treino: 26256 | Validação: 8784 | Teste: 8760
------------------------------------------------------------
✅ Divisão e Nivelamento concluídos com perfeição!
O PLD agora varia matematicamente entre: 0.0 e 1.0
------------------------------------------------------------


,PLD,CARGA_MWMED,ENA_MWMED,EAR_PERCENTUAL,PLD_lag_1,PLD_lag_2,PLD_lag_3,PLD_lag_24,CARGA_lag_1,CARGA_lag_24,ENA_lag_24,EAR_lag_24
DATA_HORA,,,,,,,,,,,,
2021-01-02 00:00:00,0.317555,0.214232,0.346486,0.037784,0.236181,0.239369,0.242248,0.216121,0.266745,0.207059,0.340558,0.034775
2021-01-02 01:00:00,0.310985,0.170623,0.346486,0.037784,0.317555,0.236181,0.239369,0.219546,0.214232,0.198038,0.340558,0.034775
2021-01-02 02:00:00,0.307993,0.142289,0.346486,0.037784,0.310985,0.317555,0.236181,0.216457,0.170623,0.182238,0.340558,0.034775
2021-01-02 03:00:00,0.310034,0.126285,0.346486,0.037784,0.307993,0.310985,0.317555,0.222873,0.142289,0.156715,0.340558,0.034775
2021-01-02 04:00:00,0.309923,0.117912,0.346486,0.037784,0.310034,0.307993,0.310985,0.220063,0.126285,0.142262,0.340558,0.034775
